In [63]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import joblib
import os

In [64]:
import_path="data/weather_aqi_data.csv"
output_path="processed/"

In [65]:
lookback=72
forecast_horizon=24

train_ratio=0.70
val_ratio=0.15

In [66]:
feature_cols=[
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "precipitation",
    "pm2_5",
    "pm10",
    "no2",
    "ozone",
    "carbon_monoxide",
]

In [67]:
target_cols=[
    "temperature",
    "humidity",
    "wind_speed",
    "aqi",
]

In [68]:
def load_data(path):
    df=pd.read_csv(path,parse_dates=["timestamp"])
    df.sort_values("timestamp",inplace=True)
    df.reset_index(drop=True,inplace=True)

    required=set(feature_cols+target_cols+["timestamp"])
    missing=required-set(df.columns)

    if missing:
        raise ValueError("Missing column in dataset: {missing}")
    return df

In [69]:
def clip_outliners(df):
    cols=feature_cols+["aqi"]
    for col in cols:
        q1=df[col].quantile(0.01)
        q99=df[col].quantile(0.99)
        before=df[col].between(q1,q99).mean()*100
        df[col]=df[col].clip(q1,q99)
    return df

In [70]:
def add_time_features(df):
    df["hour_sin"]=np.sin(2*np.pi*df["timestamp"].dt.hour  /24)
    df["hour_cos"]=np.cos(2*np.pi*df["timestamp"].dt.hour  /24)
    df["day_sin"]=np.sin(2*np.pi*df["timestamp"].dt.dayofyear    /365)
    df["day_cos"]=np.cos(2*np.pi*df["timestamp"].dt.dayofyear    /365)

    for feat in ["hour_sin","hour_cos","day_sin","day_cos"]:
        if feat not in feature_cols:
            feature_cols.append(feat)
    return df

In [71]:
def scale_data(df,train_end_idx):
    feat_scalar=MinMaxScaler(feature_range=(0,1))
    target_scalar=MinMaxScaler(feature_range=(0,1))
    
    train_feats=df[feature_cols].loc[:train_end_idx]
    target_feats=df[target_cols].loc[:train_end_idx]

    feat_scalar.fit(train_feats)
    target_scalar.fit(target_feats)

    df_feats=pd.DataFrame(feat_scalar.transform(df[feature_cols]),columns=feature_cols)
    df_targets=pd.DataFrame(target_scalar.transform(df[target_cols]),columns=target_cols)

    os.makedirs(output_path, exist_ok=True)
    joblib.dump(feat_scalar,   os.path.join(output_path, "feat_scaler.pkl"))
    joblib.dump(target_scalar, os.path.join(output_path, "target_scaler.pkl"))
    print(f"    ✓ Scalers saved to {output_path}")
 
    return df_feats.values, df_targets.values, feat_scalar, target_scalar

In [72]:
def build_sequences(features, targets,lookback,horizon):
    X,y=[],[]
    total=len(features)-lookback-horizon+1
    for i in range (total):
        X.append(features[i:i+lookback])
        y.append(targets[i+lookback:i+lookback+horizon])
    return np.array(X,dtype=np.float32), np.array(y,dtype=np.float32)

In [73]:
def split_and_save(features, targets, timestamps):
    n            = len(features)
    train_end    = int(n * train_ratio)
    val_end      = int(n * (train_ratio + val_ratio))
 
    print(" Building sliding-window sequences ...")
    X, y = build_sequences(features, targets, lookback, forecast_horizon)
    print(f"    ✓ X shape: {X.shape}   (samples, lookback, features)")
    print(f"    ✓ y shape: {y.shape}   (samples, horizon, targets)")
 
    seq_train_end = train_end - lookback - forecast_horizon
    seq_val_end   = val_end   - lookback - forecast_horizon
 
    X_train, y_train = X[:seq_train_end],              y[:seq_train_end]
    X_val,   y_val   = X[seq_train_end:seq_val_end],   y[seq_train_end:seq_val_end]
    X_test,  y_test  = X[seq_val_end:],                y[seq_val_end:]
 
    print(f"\n    Train : {X_train.shape[0]:,} samples")
    print(f"    Val   : {X_val.shape[0]:,} samples")
    print(f"    Test  : {X_test.shape[0]:,} samples")
 
    print(f"\n Saving processed arrays to {output_path} ...")
    os.makedirs(output_path, exist_ok=True)
    np.save(os.path.join(output_path, "X_train.npy"), X_train)
    np.save(os.path.join(output_path, "y_train.npy"), y_train)
    np.save(os.path.join(output_path, "X_val.npy"),   X_val)
    np.save(os.path.join(output_path, "y_val.npy"),   y_val)
    np.save(os.path.join(output_path, "X_test.npy"),  X_test)
    np.save(os.path.join(output_path, "y_test.npy"),  y_test)
    print(f"    ✓ All arrays saved.")
 
    return X_train, y_train, X_val, y_val, X_test, y_test

In [74]:
def summarize(X_train, y_train, X_val, y_val, X_test, y_test):
    print("\n" + "─" * 55)
    print("  PREPROCESSING SUMMARY")
    print("─" * 55)
    print(f"  Lookback window  : {lookback} hours")
    print(f"  Forecast horizon : {forecast_horizon} hours")
    print(f"  Input features   : {len(feature_cols)} → {feature_cols}")
    print(f"  Target variables : {len(target_cols)} → {target_cols}")
    print(f"\n  X_train : {X_train.shape}   y_train : {y_train.shape}")
    print(f"  X_val   : {X_val.shape}     y_val   : {y_val.shape}")
    print(f"  X_test  : {X_test.shape}    y_test  : {y_test.shape}")
    print("─" * 55)

In [75]:
if __name__ == "__main__":
    print("\n  Preprocessing Weather + AQI Data\n")
 
    df= load_data(import_path)
    df= clip_outliners(df)
    df= add_time_features(df)
 
    train_end_idx   = int(len(df) * train_ratio)
    features, targets, feat_scaler, target_scaler = scale_data(df, train_end_idx)
 
    X_train, y_train, X_val, y_val, X_test, y_test = split_and_save(
        features, targets, df["timestamp"].values
    )
 
    summarize(X_train, y_train, X_val, y_val, X_test, y_test)
    print("\n Done! Processed data ready in:", output_path)


  Preprocessing Weather + AQI Data

    ✓ Scalers saved to processed/
 Building sliding-window sequences ...
    ✓ X shape: (29807, 72, 14)   (samples, lookback, features)
    ✓ y shape: (29807, 24, 4)   (samples, horizon, targets)

    Train : 20,835 samples
    Val   : 4,485 samples
    Test  : 4,487 samples

 Saving processed arrays to processed/ ...
    ✓ All arrays saved.

───────────────────────────────────────────────────────
  PREPROCESSING SUMMARY
───────────────────────────────────────────────────────
  Lookback window  : 72 hours
  Forecast horizon : 24 hours
  Input features   : 14 → ['temperature', 'humidity', 'wind_speed', 'wind_direction', 'precipitation', 'pm2_5', 'pm10', 'no2', 'ozone', 'carbon_monoxide', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
  Target variables : 4 → ['temperature', 'humidity', 'wind_speed', 'aqi']

  X_train : (20835, 72, 14)   y_train : (20835, 24, 4)
  X_val   : (4485, 72, 14)     y_val   : (4485, 24, 4)
  X_test  : (4487, 72, 14)    y_test